# 02 – Ügyfélszegmentáció (Customer Segmentation)

**Bemenet:** `data/processed/online_retail_ready_for_rfm.parquet`  
**Kimenet:** `data/processed/rfm_features.parquet`, `models/scaler_rfm.joblib`

*Az adatok előkészítése a `01_data_preparation.ipynb` notebookban történt.*

**Tartalom:**

2\. 2. Feature Engineering és az adatszivárgás (data leakage) megelőzése

3\. Statisztikai outlier-kezelés és skálázás

4\. K-means klaszterezés: Szegmentálás és a szegmensek profilozása.

5\. Kiterjesztett EDA: A klaszterek vizualizálása (pl. Snake plot vagy Heatmap).

In [ ]:
# ============================================================
# Konfiguráció és könyvtárak betöltése
# ============================================================
import pandas as pd
import numpy as np
from config import (
    READY_FOR_RFM_PARQUET, RFM_FEATURES_PARQUET,
    IMAGES_DIR, MODELS_DIR, SCALER_PATH, CUTOFF_DATE
)

# Mappastruktúra biztosítása
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Előző notebook kimenetének betöltése
df = pd.read_parquet(READY_FOR_RFM_PARQUET)
print(f"Betöltve: {READY_FOR_RFM_PARQUET}")
print(f"Sorok: {len(df):,} | Oszlopok: {df.shape[1]}")

## 2. Feature Engineering és az adatszivárgás (data leakage) megelőzése

A klasszikus prediktív Customer Lifetime Value (CLV) modellezés legkritikusabb pontja az **adatszivárgás (data leakage) megelőzése**. Annak érdekében, hogy egy valós üzleti szituációt szimuláljunk (ahol a múltbeli adatokból próbáljuk megjósolni a jövőt), az adathalmazt egy időbeli vágási pont (Cutoff Date) mentén **két szigorúan elkülönülő ablakra bontjuk**:

1. **Megfigyelési ablak (X - Observation Window):** Az adathalmaz kezdete és a vágási pont közötti időszak. Kizárólag ezekből az adatokból számítjuk ki az ügyfelek RFM (Recency, Frequency, Monetary) és egyéb viselkedési mutatóit. A gépi tanulási modell csak ezt fogja látni.
2. **Célablak (y - Target Window):** A vágási ponttól az adathalmaz végéig tartó időszak (kb. az utolsó 90 nap). Itt számoljuk ki az ügyfelek tényleges értékét (a célváltozót), amit a modellnek majd prediktálnia kell.

*Kiválasztott vágási pont:* **2011. szeptember 9.** (Mivel az adathalmaz 2011. december 9-én ér véget, ez pontosan egy 90 napos (negyedéves) előrejelzési ablakot biztosít, ami iparági sztenderd a B2B/B2C szegmentációban).

In [ ]:
# ============================================================
# 2.1 - Időablakok (Time-Windows) felállítása és szétválasztása
# ============================================================

print("Időablakok szétválasztása az adatszivárgás elkerülése érdekében...")

# Vágási pont (Cutoff date) definiálása: pontosan 90 nappal az adathalmaz vége előtt
CUTOFF_DATE_TS = pd.to_datetime(CUTOFF_DATE)

# 2.1. Megfigyelési ablak (X feature-ök alapja)
df_obs = df[df['InvoiceDate'] < CUTOFF_DATE_TS].copy()

# 2.2. Célablak (y célváltozó alapja)
df_target = df[df['InvoiceDate'] >= CUTOFF_DATE_TS].copy()

print("-" * 50)
print(f"Teljes vizsgált időszak:  {df['InvoiceDate'].min().date()}  --->  {df['InvoiceDate'].max().date()}")
print(f"Vágási pont (Cutoff):     {CUTOFF_DATE_TS.date()}")
print("-" * 50)
print(f"Megfigyelési ablak (X):   {len(df_obs):,} sor")
print(f"Célablak (y):             {len(df_target):,} sor")

# 2.3 Ellenőrizzük, hogy hány olyan ügyfél van a megfigyelési ablakban, akit érdemes vizsgálni
valos_ugyfelek_szama = df_obs['Customer ID'].nunique()
print(f"\nModellezhető egyedi ügyfelek száma a megfigyelési ablakban: {valos_ugyfelek_szama:,}")

### 2.2 Kiterjesztett RFM Feature Engineering (csak az X ablakon)

A megfigyelési ablak (`df_obs`) felhasználásával kiszámítjuk a vásárlók egyedi profilját leíró mutatókat. A hagyományos RFM modellt **kiterjesztjük a visszaküldési (return) metrikákkal**, mivel a magas sztornóarány kritikus indikátora a jövőbeli lemorzsolódásnak (churn) és a csökkenő élettartam-értéknek (CLV).

**Kiszámított Feature-ök:**
* `recency_days`: Utolsó vásárlás óta eltelt napok száma (a vágási ponttól visszaszámolva).
* `frequency`: Sikeres (pozitív) vásárlási tranzakciók (Invoice) száma.
* `monetary_total`: Nettó elköltött összeg (a visszaküldések értékével csökkentve).
* `monetary_avg`: Átlagos nettó kosárérték ($monetary\_total / frequency$).
* `return_count`: Visszaküldött (sztornó) rendelések száma.
* `return_ratio`: Visszaküldések aránya az összes aktivitáshoz képest.

*Kritikus ML Best Practice:* Minden aggregációt szigorúan a `df_obs` adathalmazon hajtunk végre, így a modell semmilyen információt nem kap a jövőbeli (vágási pont utáni) viselkedésről.

In [ ]:
# ============================================================
# 2.2 - Kiterjesztett RFM metrikák kiszámítása
# ============================================================

print("Ügyfélszintű RFM és Return feature-ök kiszámítása a megfigyelési ablakból...\n")

# 2.2.1. Pozitív tranzakciók (Vásárlások) szűrése a Recency és Frequency számításhoz
purchases = df_obs[df_obs['Quantity'] > 0]

rfm = purchases.groupby('Customer ID').agg(
    # Recency: Cutoff dátum - utolsó vásárlás dátuma
    recency_days=('InvoiceDate', lambda x: (CUTOFF_DATE_TS - x.max()).days),
    # Frequency: Egyedi számlák (Invoice-ok) száma
    frequency=('Invoice', 'nunique')
)

# 2.2.2. Monetary: Nettó költés (Vásárlások + Sztornók együttes összege az ablakban)
monetary = df_obs.groupby('Customer ID')['LineTotal'].sum().rename('monetary_total')
rfm = rfm.join(monetary)

# 2.2.3. Visszaküldések (Returns) azonosítása és számolása
returns = df_obs[df_obs['Quantity'] < 0]
return_counts = returns.groupby('Customer ID')['Invoice'].nunique().rename('return_count')

# Balra csatlakozás (Left join): akinek nincs visszaküldése, annál a NaN-t 0-ra cseréljük
rfm = rfm.join(return_counts).fillna({'return_count': 0})

# 2.2.4. Származtatott (Derived) mutatók kiszámítása
rfm['monetary_avg'] = rfm['monetary_total'] / rfm['frequency']
rfm['return_ratio'] = rfm['return_count'] / (rfm['frequency'] + rfm['return_count'])

# 2.2.5. QA: Extrém esetek szűrése (pl. aki a megfigyelési ablakban összességében mínuszban van)
# (Ez előfordulhat, ha valaki csak visszaküldött az X ablakban)
rfm = rfm[rfm['monetary_total'] > 0]

print("Feature Engineering sikeresen befejeződött.")
print(f"Létrejött RFM mátrix dimenziói: {rfm.shape[0]:,} ügyfél, {rfm.shape[1]} feature")
print("-" * 50)
display(rfm.head())

## 3. Statisztikai Outlier-kezelés és skálázás

A Feature Engineering során létrehozott RFM változók jellemzően erősen jobbra ferde (right-skewed) eloszlást mutatnak: a vásárlók nagy része kis értékben és ritkán vásárol, míg egy szűk réteg extrém magas frekvenciát és költést produkál.

Mivel a következő lépésben K-means klaszterezést alkalmazunk – amely az Euklideszi távolságokra épül, és így rendkívül érzékeny a szélsőértékekre és a léptékkülönbségekre –, elengedhetetlen a változók eloszlásának vizsgálata, normalizálása és skálázása az algoritmus betanítása előtt.

### 3.1 Az eloszlások vizuális diagnosztikája
Első lépésként megvizsgáljuk az alapvető RFM mutatók (Recency, Frequency, Monetary) eloszlását boxplot ábrák segítségével, hogy felmérjük a statisztikai outlierek mértékét.

In [ ]:
# ============================================================
# 3.1 - RFM változók eloszlásának vizsgálata
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

print("RFM változók eloszlásának vizualizálása...")

# Csak az alap RFM feature-öket nézzük a diagnosztikához
features_to_plot = ['recency_days', 'frequency', 'monetary_total']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Változók Eloszlása (Nyers adatok)', fontsize=16)

for i, col in enumerate(features_to_plot):
    sns.boxplot(x=rfm[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'{col} (Boxplot)')
    axes[i].set_xlabel('')

plt.tight_layout()

# Kép mentése dokumentációs céllal
fig_path_rfm = IMAGES_DIR / "rfm_raw_distributions.png"
plt.savefig(fig_path_rfm, bbox_inches="tight")
print(f"Ábra mentve: {fig_path_rfm}")
plt.show()

# Gyors statisztika a ferdeségről (Skewness)
# Ha az érték > 1 vagy < -1, az eloszlás erősen ferde
print("\nVáltozók ferdesége (Skewness):")
display(rfm[features_to_plot].skew().to_frame(name='Skewness').round(2))

### 3.2 Log-transzformáció és standardizálás

Az extrém magas ferdeségi (Skewness) értékek miatt a K-means klaszterezés előtt az alábbi kétlépcsős adat-előkészítést hajtjuk végre:
1. **Log-transzformáció (`np.log1p`):** Megtartjuk a legértékesebb vásárlóinkat ("Bálnák"), de a logaritmikus skálázással "összenyomjuk" a kiugró értékeket, így az eloszlás közelebb kerül a normál eloszláshoz. A `log1p` (log(x+1)) használata azért biztonságos, mert kezeli a 0 értékeket is (pl. 0 napos recency).
2. **Standardizálás (`StandardScaler`):** Mivel a K-means Euklideszi távolságot számol, a változókat azonos dimenzióba (átlag = 0, szórás = 1) kell hozni. Ennek hiányában a nagyobb számosságú metrikák (pl. Monetary) elnyomnák a kisebbeket (pl. Frequency).

*Megjegyzés: A klaszterezéshez csak a hagyományos R, F, M változókat használjuk. A visszaküldési (return) metrikák később, az XGBoost modellnél kapnak főszerepet.*

In [ ]:
# ============================================================
# 3.2. - Log-transzformáció és Skálázás
# ============================================================
from sklearn.preprocessing import StandardScaler
import joblib

print("Log-transzformáció és standardizálás folyamatban...")

# Csak az R, F, M változókat klaszterezzük
rfm_features = ['recency_days', 'frequency', 'monetary_total']
rfm_cluster_data = rfm[rfm_features].copy()

# 3.2.1. Lépés: Log-transzformáció a ferdeség csökkentésére
rfm_log = np.log1p(rfm_cluster_data)

# Nézzük meg, mennyit javult a ferdeség (Skewness)
print("\nFerdeség (Skewness) a Log-transzformáció UTÁN:")
display(rfm_log.skew().to_frame(name='Skewness').round(2))

# 3.2.2. Lépés: Standardizálás
scaler = StandardScaler()
rfm_scaled_array = scaler.fit_transform(rfm_log)

# Visszaalakítjuk DataFrame-be, hogy megmaradjon a Customer ID index
rfm_scaled = pd.DataFrame(
    rfm_scaled_array, 
    index=rfm_log.index, 
    columns=rfm_features
)

# 3.2.3. Lépés: A Scaler objektum elmentése (Kritikus lépés a Streamlit miatt!)
joblib.dump(scaler, SCALER_PATH)

print("-" * 50)
print(f"✔️ StandardScaler sikeresen illesztve és mentve ide: {SCALER_PATH}")
print("A skálázott (K-means bemeneti) adatok első 5 sora:")
display(rfm_scaled.head())

In [ ]:
# ============================================================
# 3.3. - RFM feature mátrix mentése a következő notebookhoz
# ============================================================

# Az összes feature-t (nyers RFM + return metrikák + skálázott RFM) egyetlen táblában mentjük
rfm_export = rfm.copy()
rfm_export[['recency_scaled', 'frequency_scaled', 'monetary_scaled']] = rfm_scaled.values

rfm_export.to_parquet(RFM_FEATURES_PARQUET, compression="snappy")
print(f"✔️ RFM feature mátrix mentve: {RFM_FEATURES_PARQUET}")
print(f"   Dimenziók: {rfm_export.shape[0]:,} ügyfél, {rfm_export.shape[1]} oszlop")

# 4. K-means Klaszterezés

*(Következő lépés – folytatás a portfólió következő iterációjában.)*

## 4.1. Az optimális klaszterszám (K) vizuális meghatározása
Ez a kód kiszámolja a WCSS-t (Könyök-módszerhez) és a Sziluett-pontszámot 2-től 10 klaszterig.

In [ ]:
# ============================================================
# 4.1 - Az optimális klaszterszám (K) meghatározása
# ============================================================
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Beimportáljuk a szükséges útvonalakat a config.py-ból
from config import RFM_FEATURES_PARQUET, IMAGES_DIR

print("K-means futtatása 2-10 klaszterre az optimális K megtalálásához...\n")

# 1. Adatok betöltése
rfm_export = pd.read_parquet(RFM_FEATURES_PARQUET)

# 2. Csak a skálázott RFM oszlopokat vesszük ki a klaszterezéshez
scaled_columns = ['recency_scaled', 'frequency_scaled', 'monetary_scaled']
rfm_scaled = rfm_export[scaled_columns]

wcss = []
silhouette_scores = []
k_range = list(range(2, 11))

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    
    # Könyök módszerhez (Within-Cluster Sum of Square)
    wcss.append(kmeans.inertia_)
    
    # Sziluett pontszámhoz
    labels = kmeans.labels_
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))

# Eredmények vizualizálása
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Könyök-módszer ábra
axes[0].plot(k_range, wcss, marker='o', linestyle='--', color='b')
axes[0].set_title('Könyök-módszer (Elbow Method)')
axes[0].set_xlabel('Klaszterek száma (K)')
axes[0].set_ylabel('WCSS')
axes[0].grid(True, alpha=0.3)

# Sziluett-elemzés ábra
axes[1].plot(k_range, silhouette_scores, marker='o', linestyle='--', color='g')
axes[1].set_title('Sziluett-elemzés (Silhouette Score)')
axes[1].set_xlabel('Klaszterek száma (K)')
axes[1].set_ylabel('Sziluett pontszám')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Ábra mentése
fig_path_kmeans = IMAGES_DIR / "kmeans_optimal_k.png"
plt.savefig(fig_path_kmeans, bbox_inches="tight")
print(f"Ábra mentve: {fig_path_kmeans}\n")
plt.show()

# ============================================================
# Automatikus kiértékelés és indoklás
# ============================================================
# Megkeressük a legmagasabb Sziluett-pontszámhoz tartozó K értéket
best_score = max(silhouette_scores)
best_index = silhouette_scores.index(best_score)
optimal_k = k_range[best_index]

print("-" * 50)
print("📊 AUTOMATIKUS KIÉRTÉKELÉS EREDMÉNYE:")
print("-" * 50)
print(f"Az optimális klaszterszám: K = {optimal_k}")
print(f"Maximális Sziluett-pontszám: {best_score:.4f}\n")

print("INDOKLÁS:")
print(f"1. A Sziluett-elemzés statisztikailag a K={optimal_k} értéknél adja a legmagasabb pontszámot.")
print("   Ez azt jelenti, hogy az adatpontok itt illeszkednek a legszorosabban a saját ")
print("   klaszterükbe, és itt különülnek el a legjobban a többi csoporttól.")
print(f"2. A Könyök-módszer (Elbow method) ábráján általában megfigyelhető, hogy a K={optimal_k}")
print("   környékén törik meg a görbe. Ennél több klaszter létrehozása már nem hozna ")
print("   akkora hibacsökkenést (WCSS), ami indokolná a modell további bonyolítását.")
print("-" * 50)

### 4.2 Az optimális klaszterszám kiválasztása: Matematika vs. üzleti logika

A fenti automatikus kiértékelés alapján az algoritmus a **K=2** értéket javasolta, mivel a Sziluett-pontszám ott éri el az abszolút maximumot. Bár statisztikailag ez adja a legélesebb (leginkább elkülönülő) határvonalat az adathalmazban, az adattudományi projektekben a tiszta matematikát mindig össze kell hangolni az üzleti céllal.

**Miért vetjük el a K=2-t?**
Egy webáruház vagy nagykereskedés számára két ügyfélszegmens (vélhetően "sokat költők" és "keveset költők") túlságosan homogén. Nem ad elég finom felbontást ahhoz, hogy személyre szabott marketingstratégiát építsünk rá (pl. nem tudjuk megkülönböztetni a lemorzsolódó VIP ügyfeleket a friss, de ígéretes vásárlóktól).

**A Data Science kompromisszum: K=4**
Ha vizuálisan megvizsgáljuk az ábrákat, azt látjuk, hogy a statisztikának van egy nagyon erős másodlagos optimuma:
1. **Sziluett-elemzés:** K=3-nál visszaesik a pontszám, de **K=4-nél egyértelmű lokális maximum (púp)** rajzolódik ki, ami azt jelzi, hogy itt ismét egy természetes, jól elkülönülő csoportosulást találtunk.
2. **Könyök-módszer:** A WCSS hibagörbe esése a K=4 és K=5 környékén kezd el kisimulni (itt található a valódi "könyök").
3. **Domain Knowledge (Iparági sztenderd):** A 4 szegmens tökéletesen megfeleltethető a klasszikus RFM kategóriáknak (pl. *1. VIP/Bajnokok, 2. Hűséges átlagos, 3. Új/Ígéretes, 4. Lemorzsolódott*).

Fentiek alapján **felülbíráljuk a gép K=2-es javaslatát, és a végleges modellt K=4 szegmensre tanítjuk be.**

In [ ]:
# ============================================================
# 4.2 - A végleges K-means modell illesztése és mentése
# ============================================================
import joblib

OPTIMAL_K = 4  # A fenti ábrák és az üzleti logika alapján választva

print(f"Végleges K-means modell betanítása K={OPTIMAL_K} értékkel...")
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans_final.fit(rfm_scaled)

# Klaszter címkék hozzáadása az eredeti export adathalmazhoz
rfm_export['cluster'] = kmeans_final.labels_

# Modell mentése a produkciós pipeline-hoz (Streamlit/API)
KMEANS_PATH = MODELS_DIR / "kmeans_rfm.joblib"
joblib.dump(kmeans_final, KMEANS_PATH)

print(f"✔️ K-means modell mentve ide: {KMEANS_PATH}")

# Ügyfelek eloszlása a klaszterekben
cluster_counts = rfm_export['cluster'].value_counts().sort_index()
print("\nÜgyfelek száma klaszterenként:")
display(cluster_counts.to_frame('Ügyfélszám'))

## 4.3. Kirajzolja a vásárlókat a 3D térben, a klaszterközéppontokat (centroidokat) pedig piros X-szel jelöli

In [ ]:
# ============================================================
# 4.3 - Klaszterek 3D térbeli vizualizációja centroidokkal (Interaktív Plotly)
# ============================================================
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

print("Klaszterek és centroidok interaktív 3D vizualizálása...")

# Üzleti címkék hozzárendelése a klaszterekhez
segment_map = {
    0: 'Lemorzsolódó / Alvó',
    1: 'Elvesztett / Inaktív',
    2: 'VIP Bajnokok',
    3: 'Új / Ígéretes'
}

# Létrehozunk egy ideiglenes DataFrame-et a Plotly számára
df_viz = rfm_scaled.copy()
df_viz['Segment'] = kmeans_final.labels_
df_viz['Segment'] = df_viz['Segment'].map(segment_map)

# Egyedi színpaletta a kérésnek megfelelően
custom_colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']

# 1. Az adatpontok ábrázolása (scatter 3d)
fig = px.scatter_3d(
    df_viz, 
    x='recency_scaled', 
    y='frequency_scaled', 
    z='monetary_scaled',
    color='Segment',
    opacity=0.6,  # MÓDOSÍTVA: 0.5-0.7 közötti átlátszóság a sűrűség érzékeltetésére
    title='Ügyfélszegmensek 3D térben',
    labels={
        'recency_scaled': 'Recency (Skálázott)',
        'frequency_scaled': 'Frequency (Skálázott)',
        'monetary_scaled': 'Monetary (Skálázott)'
    },
    color_discrete_sequence=custom_colors # MÓDOSÍTVA: A megadott színkódok használata
)

# MÓDOSÍTVA: Eltávolítottuk vagy teljesen átlátszóvá tettük a pontok fekete keretét, 
# hogy az "alpha" hatás (egymásra rétegződés) jobban érvényesüljön a sűrűbb részeken.
fig.update_traces(
    marker=dict(line=dict(width=0))
)

# 2. A Centroidok látványos hozzáadása
centroids = kmeans_final.cluster_centers_

# A centroidok trace-ét a pontok UTÁN adjuk hozzá, ami Plotly esetében 
# segíti, hogy a többi réteg felett ("legmagasabb Z-order") jelenjen meg.
fig.add_trace(go.Scatter3d(
    x=centroids[:, 0],
    y=centroids[:, 1],
    z=centroids[:, 2],
    mode='markers',
    marker=dict(
        size=16,
        color='#000000',          # MÓDOSÍTVA: Fekete szín
        symbol='diamond',         # MÓDOSÍTVA: Gyémánt alak
        opacity=1.0,              # MÓDOSÍTVA: 100% átlátszatlan
        line=dict(width=2, color='#FFFFFF') # MÓDOSÍTVA: Vékony fehér kontrasztkeret
    ),
    name='Klaszter Középpontok (Centroidok)'
))

# Sablon és layout beállítások
fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=40), 
    legend_title_text='Ügyfélszegmensek',
    template='plotly_white' 
)

fig.show()

# HTML mentése portfólióhoz
# Feltételezem, az IMAGES_DIR már definiálva van a kódod elején
fig_path_3d_html = IMAGES_DIR / "kmeans_3d_clusters_interactive.html"
fig.write_html(str(fig_path_3d_html))
print(f"✔️ Interaktív HTML ábra mentve ide: {fig_path_3d_html}")

### 4.4 - Klaszterek 2D páronkénti vizualizációja

Bár a 3D-s ábra kiválóan szemlélteti a szegmensek térbeli elhelyezkedését, az üzleti értelmezhetőség érdekében érdemes az RFM változókat páronként is megvizsgálni. Az alábbi vizualizáció célzottan a három legfontosabb összefüggést mutatja be:

1. **Recency vs. Frequency:** Megmutatja az ügyfelek aktivitási mintázatát. Jól látható az "Új/Ígéretes" (alacsony Recency, de még alacsony Frequency) és a "Lemorzsolódó" (magas Recency, alacsony Frequency) szegmensek közötti éles határvonal.
2. **Recency vs. Monetary:** Kiemeli, hogy a legfrissebb vásárlóink egyben a legértékesebbek-e. A "VIP Bajnokok" szegmens egyértelműen dominálja a bal felső sarkot (legújabb vásárlás, legmagasabb költés).
3. **Frequency vs. Monetary:** A lojalitás és az értékteremtés kapcsolatát ábrázolja. Ez az ábra bizonyítja a klasszikus üzleti tételt: a gyakoribb vásárlások szinte kivétel nélkül magasabb összesített bevételt (Monetary) eredményeznek.

In [ ]:
# ============================================================
# 4.4 - Klaszterek 2D Páronkénti vizualizációja (Releváns Subplotok)
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, display  # Hozzáadva a Notebookos megjelenítéshez

print("Klaszterek 2D páronkénti ábrázolása (Statikus egyedi diagramok)...")

# A már korábban definiált egyedi színpaletta és szegmensek
custom_colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
segments = list(segment_map.values())

# 1x3-as subplot rács létrehozása
fig_2d = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Recency vs. Frequency', 'Recency vs. Monetary', 'Frequency vs. Monetary'),
    horizontal_spacing=0.08
)

# Iterálunk a szegmenseken és a hozzájuk tartozó színeken
for i, segment_name in enumerate(segments):
    # Szűrés az adott szegmensre
    df_sub = df_viz[df_viz['Segment'] == segment_name]
    color = custom_colors[i]
    
    # 1. Ábra: Recency vs. Frequency
    fig_2d.add_trace(
        go.Scatter(
            x=df_sub['recency_scaled'], y=df_sub['frequency_scaled'],
            mode='markers', name=segment_name, legendgroup=segment_name,
            marker=dict(color=color, opacity=0.6, line=dict(width=0))
        ),
        row=1, col=1
    )
    
    # 2. Ábra: Recency vs. Monetary
    fig_2d.add_trace(
        go.Scatter(
            x=df_sub['recency_scaled'], y=df_sub['monetary_scaled'],
            mode='markers', name=segment_name, legendgroup=segment_name, showlegend=False,
            marker=dict(color=color, opacity=0.6, line=dict(width=0))
        ),
        row=1, col=2
    )
    
    # 3. Ábra: Frequency vs. Monetary
    fig_2d.add_trace(
        go.Scatter(
            x=df_sub['frequency_scaled'], y=df_sub['monetary_scaled'],
            mode='markers', name=segment_name, legendgroup=segment_name, showlegend=False,
            marker=dict(color=color, opacity=0.6, line=dict(width=0))
        ),
        row=1, col=3
    )

# Tengelyfeliratok és letisztult kinézet frissítése
fig_2d.update_layout(
    template='plotly_white',
    title_text='Ügyfélszegmensek 2D nézetben - RFM Változók Kapcsolata',
    margin=dict(l=40, r=20, b=40, t=80),
    legend_title_text='Ügyfélszegmensek',
    height=450 # Kompaktabb magasság a Notebookban való megjelenítéshez
)

# X és Y tengelyek feliratai minden ábránál
fig_2d.update_xaxes(title_text="Recency (Skálázott)", row=1, col=1)
fig_2d.update_yaxes(title_text="Frequency (Skálázott)", row=1, col=1)

fig_2d.update_xaxes(title_text="Recency (Skálázott)", row=1, col=2)
fig_2d.update_yaxes(title_text="Monetary (Skálázott)", row=1, col=2)

fig_2d.update_xaxes(title_text="Frequency (Skálázott)", row=1, col=3)
fig_2d.update_yaxes(title_text="Monetary (Skálázott)", row=1, col=3)

# 1. Kép mentése statikus PNG formátumban (kiváló minőségben prezentációhoz)
# Megjegyzés: a mentéshez a 'kaleido' csomag telepítése szükséges! (pip install kaleido)
fig_path_2d_png = IMAGES_DIR / "kmeans_2d_subplots.png"
fig_2d.write_image(str(fig_path_2d_png), width=1400, height=500, scale=2)
print(f"✔️ Statikus 2D ábra (PNG) mentve ide: {fig_path_2d_png}\n")

# 2. Megjelenítés a Jupyter Notebookban közvetlenül a lementett képből
print("📊 Ábra megjelenítése:")
display(Image(filename=str(fig_path_2d_png)))

# (Opcionális: Ha az interaktív Plotly verziót is meg akarod próbálni megjeleníteni, vedd ki a kommentből a lenti sort)
# fig_2d.show()

## 5. Kiterjesztett EDA: Klaszterek vizualizálása (Snake Plot)
A "Snake Plot" (kígyó ábra) a marketing analitika klasszikus eszköze a klaszterek profilozására. Ez megmutatja, hogy az egyes csoportok az átlagtól milyen irányba (pozitív/negatív) és milyen mértékben térnek el a standardizált RFM skálán.

**(Tipp: Amikor ez lefut, a táblázat alapján már el is lehet nevezni a klasztereket. Például, ha a 0-s klaszternek kicsi a recency értéke, de magas a frequency és a monetary, ők a "VIP Bajnokok".)**

In [ ]:
# ============================================================
# 5. Kiterjesztett EDA: Klaszterek vizualizálása (Snake Plot)
# ============================================================
import seaborn as sns

print("Klaszterek profilozása Snake Plot segítségével...")

rfm_scaled_viz = rfm_scaled.copy()
# Beszédes nevek beemelése
rfm_scaled_viz['Segment'] = pd.Series(kmeans_final.labels_, index=rfm_scaled.index).map(segment_map)

rfm_features_list = ['recency_scaled', 'frequency_scaled', 'monetary_scaled']
rfm_melted = pd.melt(
    rfm_scaled_viz.reset_index(),
    id_vars=['Customer ID', 'Segment'],
    value_vars=rfm_features_list,
    var_name='Metrika',
    value_name='Standardizált Érték'
)

plt.figure(figsize=(10, 6))
# A hue most már a 'Segment' oszlopot használja!
sns.lineplot(x='Metrika', y='Standardizált Érték', hue='Segment', data=rfm_melted, palette='tab10', marker='o', linewidth=2)
plt.title('Ügyfélszegmensek profilja (Snake Plot)', fontsize=14)
plt.legend(title='Szegmens', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

fig_path_snake = IMAGES_DIR / "kmeans_snake_plot.png"
plt.savefig(fig_path_snake, bbox_inches="tight")
print(f"Ábra mentve: {fig_path_snake}")
plt.show()

# Címkézzük fel az eredeti adatokat is!
rfm_export['Segment'] = rfm_export['cluster'].map(segment_map)

cluster_profile = rfm_export.groupby('Segment')[['recency_days', 'frequency', 'monetary_total', 'return_ratio']].mean().round(2)
print("\nKlaszterek átlagos jellemzői (Nyers adatok alapján):")
display(cluster_profile)

## 5.2 Szegmentált adatok mentése az XGBoost számára
Az utolsó lépés ebben a notebookban, hogy a most létrehozott cluster oszlopot (és a kiterjesztett RFM változókat, mint a return_ratio) kimentsük, ami majd bemenetként szolgál a 03-as notebookba.

In [ ]:
# ============================================================
# 5.2 - Szegmentált adatok mentése az XGBoost számára
# ============================================================
from config import PROCESSED_DIR

CUSTOMER_SEGMENTS_PARQUET = PROCESSED_DIR / "customer_segments.parquet"

# Kimentjük az R,F,M és az új 'Segment' oszlopokat is
rfm_export.to_parquet(CUSTOMER_SEGMENTS_PARQUET, compression="snappy")
print(f"✔️ Klaszter címkékkel gazdagított adathalmaz mentve: {CUSTOMER_SEGMENTS_PARQUET}")
print(f"   Dimenziók: {rfm_export.shape[0]:,} ügyfél, {rfm_export.shape[1]} oszlop")